In [7]:
import os
import pandas as pd
import numpy as np
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from scipy.sparse import hstack, save_npz
import joblib
from tqdm import tqdm

# Ensure VADER lexicon is downloaded
nltk.download('vader_lexicon', quiet=True)

# 1. Setup Paths
INPUT_CSV = "../data/text/final_text_data.csv"
OUTPUT_MATRIX = "../data/final_text_features.npz"
MODEL_DIR = "../models/"

# Ensure the models directory exists so we can save our Vectorizer later
os.makedirs(MODEL_DIR, exist_ok=True)

print("Loading cleaned text data...")
df = pd.read_csv(INPUT_CSV)

# Safety check: Fill any empty rows with an empty string to prevent crashes
# FIX: Using the 'content' column
df['content'] = df['content'].fillna("")

# --- STEP 1: SMART TF-IDF (The Bigram Upgrade) ---
print("Applying Smart TF-IDF (Extracting Unigrams & Bigrams)...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# This returns a highly compressed "Sparse Matrix"
X_tfidf = tfidf.fit_transform(df['content'])

# --- STEP 2: VADER SENTIMENT EXTRACTION ---
print("Calculating VADER Sentiment Scores...")
sia = SentimentIntensityAnalyzer()

# Calculate sentiment scores using the 'content' column
sentiment_scores = [sia.polarity_scores(text)['compound'] for text in tqdm(df['content'])]

# Convert the flat list into a 2D vertical column
X_sentiment = np.array(sentiment_scores).reshape(-1, 1)

# --- STEP 3: COMBINING FEATURES EFFICIENTLY ---
print("Merging TF-IDF and Sentiment features...")
X_final = hstack([X_tfidf, X_sentiment])

# --- STEP 4: SAVING ARTIFACTS ---
print("Saving features and vectorizer model...")

# Save the target labels so we can use them in training (Phase 5)
# FIX: Using the 'risk_level' column and saving directly to the data folder
df['risk_level'].to_csv("../data/text_labels.csv", index=False)

# Save the feature matrix as a compressed .npz file
save_npz(OUTPUT_MATRIX, X_final)

# CRITICAL: Save the TF-IDF vocabulary rules!
joblib.dump(tfidf, os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))

print(f"Success! Final text feature matrix shape: {X_final.shape}")
print("Saved features to:", OUTPUT_MATRIX)
print("Saved Vectorizer to:", os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))

Loading cleaned text data...
Applying Smart TF-IDF (Extracting Unigrams & Bigrams)...
Calculating VADER Sentiment Scores...


100%|██████████| 7102/7102 [00:01<00:00, 5888.54it/s]


Merging TF-IDF and Sentiment features...
Saving features and vectorizer model...
Success! Final text feature matrix shape: (7102, 5001)
Saved features to: ../data/final_text_features.npz
Saved Vectorizer to: ../models/tfidf_vectorizer.pkl
